In [0]:
-- shared_region_crosswalk.sql

-- 1. Create only if the table does not exist.
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.shared_region_crosswalk (
  country STRING,
  source_region STRING,
  region_code STRING,
  region_name STRING,
  valid_from_year INT,
  valid_to_year INT,
  source_code STRING,
  geographic_level STRING
);

-- 2. Maintain validated aliases without deleting existing rows.
MERGE INTO pondolens.dev_silver.shared_region_crosswalk AS t
USING (
  SELECT
    'Philippines' AS country,
    source_region,
    region_code,
    region_name,
    valid_from_year,
    valid_to_year,
    source_code,
    geographic_level
  FROM VALUES
    (
      'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM)',
      'BARMM',
      'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO',
      2022, 2025, 'BARMM', 'region'
    ),
    (
      'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM/ARMM)',
      'BARMM',
      'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO',
      2021, 2021, 'BARMM/ARMM', 'region'
    ),
    (
      'CORDILLERA ADMINISTRATIVE REGION (CAR)',
      'CAR', 'CORDILLERA ADMINISTRATIVE REGION',
      2021, 2025, 'CAR', 'region'
    ),
    (
      'NATIONAL CAPITAL REGION (NCR)',
      'NCR', 'NATIONAL CAPITAL REGION',
      2021, 2025, 'NCR', 'region'
    ),
    (
      'NEGROS ISLAND REGION (NIR)',
      'NIR', 'NEGROS ISLAND REGION',
      2024, 2025, 'NIR', 'region'
    ),
    (
      'REGION I (ILOCOS REGION)',
      'I', 'ILOCOS REGION',
      2021, 2025, 'I', 'region'
    ),
    (
      'REGION II (CAGAYAN VALLEY)',
      'II', 'CAGAYAN VALLEY',
      2021, 2025, 'II', 'region'
    ),
    (
      'REGION III (CENTRAL LUZON)',
      'III', 'CENTRAL LUZON',
      2021, 2025, 'III', 'region'
    ),
    (
      'REGION IV-A (CALABARZON)',
      'IVA', 'CALABARZON',
      2024, 2025, 'IV-A', 'region'
    ),
    (
      'REGION IVA (CALABARZON)',
      'IVA', 'CALABARZON',
      2022, 2024, 'IVA', 'region'
    ),
    (
      'REGION IVA CALABARZON',
      'IVA', 'CALABARZON',
      2021, 2021, 'IVA', 'region'
    ),
    (
      'MIMAROPA REGION',
      'IVB', 'MIMAROPA REGION',
      2021, 2025, CAST(NULL AS STRING), 'region'
    ),
    (
      'REGION V (BICOL REGION)',
      'V', 'BICOL REGION',
      2021, 2025, 'V', 'region'
    ),
    (
      'REGION VI (WESTERN VISAYAS)',
      'VI', 'WESTERN VISAYAS',
      2021, 2025, 'VI', 'region'
    ),
    (
      'REGION VII (CENTRAL VISAYAS)',
      'VII', 'CENTRAL VISAYAS',
      2021, 2025, 'VII', 'region'
    ),
    (
      'REGION VIII (EASTERN VISAYAS)',
      'VIII', 'EASTERN VISAYAS',
      2021, 2025, 'VIII', 'region'
    ),
    (
      'REGION IX (ZAMBOANGA PENINSULA)',
      'IX', 'ZAMBOANGA PENINSULA',
      2021, 2025, 'IX', 'region'
    ),
    (
      'REGION X (NORTHERN MINDANAO)',
      'X', 'NORTHERN MINDANAO',
      2021, 2025, 'X', 'region'
    ),
    (
      'REGION XI (DAVAO REGION)',
      'XI', 'DAVAO REGION',
      2021, 2025, 'XI', 'region'
    ),
    (
      'REGION XII (SOCCSKSARGEN)',
      'XII', 'SOCCSKSARGEN',
      2021, 2025, 'XII', 'region'
    ),
    (
      'REGION XIII (CARAGA)',
      'XIII', 'CARAGA',
      2021, 2025, 'XIII', 'region'
    ),
    (
      'CENTRAL OFFICE',
      'CO', 'CENTRAL OFFICE',
      2021, 2025, CAST(NULL AS STRING), 'other'
    )
  AS mapping (
    source_region,
    region_code,
    region_name,
    valid_from_year,
    valid_to_year,
    source_code,
    geographic_level
  )
) AS s

ON LOWER(TRIM(t.country)) = LOWER(s.country)
AND TRIM(t.source_region) = s.source_region

WHEN MATCHED THEN UPDATE SET
  t.region_code = s.region_code,
  t.region_name = s.region_name,
  t.valid_from_year = LEAST(t.valid_from_year, s.valid_from_year),
  t.valid_to_year = GREATEST(t.valid_to_year, s.valid_to_year),
  t.source_code = s.source_code,
  t.geographic_level = s.geographic_level

WHEN NOT MATCHED THEN INSERT (
  country,
  source_region,
  region_code,
  region_name,
  valid_from_year,
  valid_to_year,
  source_code,
  geographic_level
)
VALUES (
  s.country,
  s.source_region,
  s.region_code,
  s.region_name,
  s.valid_from_year,
  s.valid_to_year,
  s.source_code,
  s.geographic_level
);

-- 3. Fill missing geographic levels for existing older aliases.
UPDATE pondolens.dev_silver.shared_region_crosswalk
SET geographic_level = CASE
  WHEN region_code IN ('PHL', 'NW') THEN 'nationwide'
  WHEN region_code = 'CO' THEN 'other'
  WHEN region_code IN (
    'BARMM', 'CAR', 'NCR', 'NIR',
    'I', 'II', 'III', 'IVA', 'IVB',
    'V', 'VI', 'VII', 'VIII', 'IX',
    'X', 'XI', 'XII', 'XIII'
  ) THEN 'region'
  ELSE geographic_level
END
WHERE LOWER(TRIM(country)) = 'philippines'
  AND geographic_level IS NULL;

-- 4. Validation: expected missing_required_fields = 0.
-- source_code may be NULL when the raw label has no code.
SELECT
  COUNT(*) AS total_rows,
  SUM(
    CASE
      WHEN source_region IS NULL
        OR region_code IS NULL
        OR region_name IS NULL
        OR geographic_level IS NULL
        OR valid_from_year IS NULL
        OR valid_to_year IS NULL
        OR valid_from_year > valid_to_year
      THEN 1 ELSE 0
    END
  ) AS missing_required_fields
FROM pondolens.dev_silver.shared_region_crosswalk
WHERE LOWER(TRIM(country)) = 'philippines';

-- 5. Preview all aliases, including CO and national records.
SELECT *
FROM pondolens.dev_silver.shared_region_crosswalk
ORDER BY geographic_level, region_code, source_region;